In [1]:
import json

import cmasher as cmr
import gc_utils
import h5py
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
from astropy.stats import biweight_location
from matplotlib import cm
from matplotlib.animation import PillowWriter
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import LogLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d, zoom
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.stats import binned_statistic_2d

In [35]:
sim = "m12i"
snap_lim = 46

tbin_num_i = 6

# about 2 minutes to run on 101 iterations
it_min = 0
it_max = 100

sim_dir = "/Users/z5114326/Documents/simulations/"

pub_data = sim_dir + "snapshot_times_public.txt"
pub_snaps = pd.read_table(pub_data, comment="#", header=None, sep=r"\s+")
pub_snaps.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
snap_lst = pub_snaps["index"].to_numpy()
time_lst = pub_snaps["time_Gyr"].to_numpy()

snap_lst = pub_snaps[pub_snaps["index"] >= snap_lim]["index"].to_numpy()
time_lst = pub_snaps[pub_snaps["index"] >= snap_lim]["time_Gyr"].to_numpy()

all_data = sim_dir + "/" + sim + "/" + sim + "_res7100/" + "snapshot_times.txt"
all_snaps = pd.read_table(all_data, comment="#", header=None, sep=r"\s+")
all_snaps.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]

proc_file = sim_dir + sim + "/" + sim + "_processed.hdf5"
proc_data = h5py.File(proc_file, "r")  # open processed data file

grp_file = sim_dir + sim + "/" + "gc_groups.json"
with open(grp_file, "r") as file:
    grp_dict = json.load(file)

ex_grp_lst = np.array([int(grp) for grp in grp_dict[sim].keys() if int(grp) != 0])
ex_grp_lst

array([ 1920378,  4414957,  8580896, 13687078, 16199669, 19898495])

In [75]:
def get_it_dict(time_lst, ex_grp_lst, tbin_con_dict):
    it_dict = {gc_utils.iteration_name(it): {} for it in range(it_min, it_max + 1)}

    for it_id in it_dict.keys():
        snp_dat_600 = proc_data[it_id]["snapshots"]["snap600"]
        gc_sur_lst = snp_dat_600["gc_id"][()]
        gp_sur_lst = snp_dat_600["group_id"][()]

        gc_dict = {gcid: {} for gcid, grp in zip(gc_sur_lst, gp_sur_lst) if grp in ex_grp_lst or grp == 0}

        it_dict[it_id] = gc_dict

    for it_id in it_dict.keys():
        for gcid in it_dict[it_id].keys():
            it_dict[it_id][gcid]["et_norm"] = []
            it_dict[it_id][gcid]["jr"] = []
            it_dict[it_id][gcid]["jp"] = []
            it_dict[it_id][gcid]["jz"] = []
            it_dict[it_id][gcid]["r3d"] = []
            it_dict[it_id][gcid]["r2d"] = []
            it_dict[it_id][gcid]["z"] = []

    for it_id in it_dict.keys():
        src_dat = proc_data[it_id]["source"]
        ana_msk = src_dat["analyse_flag"][()] == 1

        for gcid in it_dict[it_id].keys():
            gcid_idx = np.where(src_dat["gc_id"][ana_msk] == gcid)[0][0]
            gcid_grp = src_dat["group_id"][ana_msk][gcid_idx]

            if gcid_grp == 0:
                tvar = "form_time"
                grp_type = "in_situ"
                tbin_con = "tbin_con_i"
            else:
                tvar = "t_acc"
                grp_type = "ex_situ"
                tbin_con = "tbin_con_e"

            tval = src_dat[tvar][ana_msk][gcid_idx]
            it_dict[it_id][gcid]["tval"] = tval
            it_dict[it_id][gcid]["grp_type"] = grp_type

            print(it_id, gcid, gcid_idx)
            it_dict[it_id][gcid]["tval_grp"] = tbin_con_dict[tbin_con][tval]

    for it_id in it_dict.keys():
        for snap in snap_lst:
            snap_id = gc_utils.snapshot_name(snap)

            snp_dat = proc_data[it_id]["snapshots"][snap_id]
            acc_msk = snp_dat["now_accreted"][()] == 1
            bnd_msk = snp_dat["bound_flag"][()] == 1

            gc_snp_lst = snp_dat["gc_id"][acc_msk & bnd_msk]

            for gcid in it_dict[it_id].keys():
                if gcid not in gc_snp_lst:
                    it_dict[it_id][gcid]["et_norm"].append(0)
                    it_dict[it_id][gcid]["jr"].append(0)
                    it_dict[it_id][gcid]["jp"].append(0)
                    it_dict[it_id][gcid]["jz"].append(0)
                    it_dict[it_id][gcid]["r3d"].append(np.nan)
                    it_dict[it_id][gcid]["r2d"].append(np.nan)
                    it_dict[it_id][gcid]["z"].append(np.nan)

                else:
                    gcid_idx = np.where(snp_dat["gc_id"][acc_msk & bnd_msk] == gcid)[0][0]
                    et_norm = snp_dat["et_norm"][acc_msk & bnd_msk][gcid_idx]
                    j_cyl = snp_dat["j.cyl"][acc_msk & bnd_msk][gcid_idx]
                    pos_cyl = snp_dat["pos.cyl"][acc_msk & bnd_msk][gcid_idx]
                    r3d = snp_dat["r"][acc_msk & bnd_msk][gcid_idx]

                    it_dict[it_id][gcid]["et_norm"].append(et_norm)
                    it_dict[it_id][gcid]["jr"].append(j_cyl[0])
                    it_dict[it_id][gcid]["jp"].append(j_cyl[1])
                    it_dict[it_id][gcid]["jz"].append(j_cyl[2])
                    it_dict[it_id][gcid]["r3d"].append(r3d)
                    it_dict[it_id][gcid]["r2d"].append(pos_cyl[0])
                    it_dict[it_id][gcid]["z"].append(pos_cyl[2])

    for it_id in it_dict.keys():
        for gcid in it_dict[it_id].keys():
            it_dict[it_id][gcid]["det_norm"] = np.diff(it_dict[it_id][gcid]["et_norm"])
            it_dict[it_id][gcid]["djr"] = np.diff(it_dict[it_id][gcid]["jr"])
            it_dict[it_id][gcid]["djp"] = np.diff(it_dict[it_id][gcid]["jp"])
            it_dict[it_id][gcid]["djz"] = np.diff(it_dict[it_id][gcid]["jz"])

            all_zeros = not np.any(it_dict[it_id][gcid]["det_norm"])
            if all_zeros:
                it_dict[it_id][gcid]["det_norm_path"] = np.nan
                it_dict[it_id][gcid]["det_norm_net"] = np.nan

                it_dict[it_id][gcid]["djr_path"] = np.nan
                it_dict[it_id][gcid]["djr_net"] = np.nan

                it_dict[it_id][gcid]["djp_path"] = np.nan
                it_dict[it_id][gcid]["djp_net"] = np.nan

                it_dict[it_id][gcid]["djz_path"] = np.nan
                it_dict[it_id][gcid]["djz_net"] = np.nan

                it_dict[it_id][gcid]["first_term"] = 0
                it_dict[it_id][gcid]["last_term"] = 0
                continue

            first_term_idx = np.min(np.nonzero(it_dict[it_id][gcid]["et_norm"]))
            last_term_idx = np.max(np.nonzero(it_dict[it_id][gcid]["et_norm"]))

            it_dict[it_id][gcid]["det_norm"][first_term_idx - 1] = 0
            it_dict[it_id][gcid]["djr"][first_term_idx - 1] = 0
            it_dict[it_id][gcid]["djp"][first_term_idx - 1] = 0
            it_dict[it_id][gcid]["djz"][first_term_idx - 1] = 0

            # if GC does not survive to z = 0
            # this just here as an ensurance as by selection they survive to z = 0
            if last_term_idx < len(it_dict[it_id][gcid]["det_norm"]) - 1:
                it_dict[it_id][gcid]["det_norm"][last_term_idx] = 0
                it_dict[it_id][gcid]["djr"][last_term_idx] = 0
                it_dict[it_id][gcid]["djp"][last_term_idx] = 0
                it_dict[it_id][gcid]["djz"][last_term_idx] = 0

            det_norm_net = (
                it_dict[it_id][gcid]["et_norm"][last_term_idx]
                - it_dict[it_id][gcid]["et_norm"][first_term_idx]
            )
            djr_net = it_dict[it_id][gcid]["jr"][last_term_idx] - it_dict[it_id][gcid]["jr"][first_term_idx]
            djp_net = it_dict[it_id][gcid]["jp"][last_term_idx] - it_dict[it_id][gcid]["jp"][first_term_idx]
            djz_net = it_dict[it_id][gcid]["jz"][last_term_idx] - it_dict[it_id][gcid]["jz"][first_term_idx]

            it_dict[it_id][gcid]["det_norm_path"] = np.sum(np.abs(it_dict[it_id][gcid]["det_norm"]))
            it_dict[it_id][gcid]["det_norm_net"] = np.abs(det_norm_net)

            it_dict[it_id][gcid]["djr_path"] = np.sum(np.abs(it_dict[it_id][gcid]["djr"]))
            it_dict[it_id][gcid]["djr_net"] = np.abs(djr_net)

            it_dict[it_id][gcid]["djp_path"] = np.sum(np.abs(it_dict[it_id][gcid]["djp"]))
            it_dict[it_id][gcid]["djp_net"] = np.abs(djp_net)

            it_dict[it_id][gcid]["djz_path"] = np.sum(np.abs(it_dict[it_id][gcid]["djz"]))
            it_dict[it_id][gcid]["djz_net"] = np.abs(djz_net)

            it_dict[it_id][gcid]["first_term"] = first_term_idx
            it_dict[it_id][gcid]["last_term"] = last_term_idx

    # add averages
    tdiff = np.diff(time_lst)
    for it_id in it_dict.keys():
        for gcid in it_dict[it_id].keys():
            r3d = np.array(it_dict[it_id][gcid]["r3d"][:-1])
            r3d_msk = ~np.isnan(r3d)
            num = np.nansum(r3d[r3d_msk] * tdiff[r3d_msk]) if len(r3d_msk) > 0 else np.nan
            den = np.sum(tdiff[r3d_msk])
            it_dict[it_id][gcid]["r3d_avg"] = num / den if den > 0 else np.nan

            r2d = np.array(it_dict[it_id][gcid]["r2d"][:-1])
            r2d_msk = ~np.isnan(r2d)
            num = np.nansum(r2d[r2d_msk] * tdiff[r2d_msk]) if len(r2d_msk) > 0 else np.nan
            den = np.sum(tdiff[r2d_msk])
            it_dict[it_id][gcid]["r2d_avg"] = num / den if den > 0 else np.nan

            z = np.array(it_dict[it_id][gcid]["z"][:-1])
            z_msk = ~np.isnan(z)
            num = np.nansum(z[z_msk] * tdiff[z_msk]) if len(z_msk) > 0 else np.nan
            den = np.sum(tdiff[z_msk])
            it_dict[it_id][gcid]["z_avg"] = num / den if den > 0 else np.nan

    return it_dict

In [45]:
def get_tcon(tunq, group_type):
    if group_type == "in_situ":
        vmin, vmax = np.min(tunq), np.max(tunq)
        bin_edges = np.linspace(vmin, vmax, tbin_num_i + 1)
        bin_indices = np.digitize(tunq, bin_edges, right=False) - 1
        bin_indices[bin_indices == tbin_num_i] = tbin_num_i - 1

        tbin_con = {t: b for t, b in zip(tunq, bin_indices)}

        t_dict = {}
        for i in range(tbin_num_i):
            t_dict[i] = {}
            t_dict[i]["edge_l"] = bin_edges[i]
            t_dict[i]["edge_u"] = bin_edges[i + 1]
    else:
        t_dict = {}
        for i in range(len(tunq)):
            t_dict[i] = {}
            t_dict[i]["edge_l"] = tunq[i]
            t_dict[i]["edge_u"] = tunq[i]

        tbin_con = {t: b for t, b in zip(tunq, t_dict.keys())}

    return t_dict, tbin_con

In [46]:
tgrp_i = np.array([])
tgrp_e = np.array([])

tvar_i = "form_time"
tvar_e = "t_acc"

for it_id in proc_data.keys():
    snp_dat_600 = proc_data[it_id]["snapshots"]["snap600"]
    src_dat = proc_data[it_id]["source"]
    ana_msk = src_dat["analyse_flag"][()] == 1

    gc_src = src_dat["gc_id"][ana_msk]

    grp_msk_600_i = snp_dat_600["group_id"][()] == 0
    gc_sur_i = snp_dat_600["gc_id"][grp_msk_600_i]
    gc_msk_i = np.isin(gc_src, gc_sur_i)
    tit_i = src_dat[tvar_i][ana_msk]
    tit_i = tit_i[gc_msk_i]
    tgrp_i = np.concatenate((tgrp_i, tit_i))

    grp_msk_600_e = np.isin(snp_dat_600["group_id"][()], ex_grp_lst)
    gc_sur_e = snp_dat_600["gc_id"][grp_msk_600_e]
    gc_msk_e = np.isin(gc_src, gc_sur_e)
    tit_e = src_dat[tvar_e][ana_msk]
    tit_e = tit_e[gc_msk_e]
    tgrp_e = np.concatenate((tgrp_e, tit_e))

t_dict_i, tbin_con_i = get_tcon(np.unique(tgrp_i), "in_situ")
t_dict_e, tbin_con_e = get_tcon(np.unique(tgrp_e), "ex_situ")

tbin_con_dict = {"tbin_con_i": tbin_con_i, "tbin_con_e": tbin_con_e}

In [47]:
tbin_con_dict["tbin_con_e"]

{2.359: 0, 3.608: 1, 5.395: 2, 7.822: 3, 9.148: 4, 11.161: 5}

In [ ]:
it_dict = get_it_dict(time_lst, ex_grp_lst, tbin_con_dict)

it000 57057454 94
it000 49867717 117
it000 55759176 126
it000 46186812 184
it000 122776884 337
it000 123320985 398
it000 116120128 420
it000 59979388 451
it000 55423942 452
it000 120191241 487
it000 125266350 507
it000 120023067 509
it000 122603231 519
it000 130489547 539
it000 123142986 598
it000 52458546 611
it000 56897591 640
it000 43418483 718
it000 40439938 726
it000 41228010 770
it000 38644983 779
it000 40235324 780
it000 50619748 785
it000 63157402 813
it000 60812016 828
it000 37454970 839
it000 62406382 847
it000 31197781 867
it000 31007272 876
it000 47163753 881
it000 47167726 914
it000 120387214 1019
it000 131619861 1134
it000 123493678 1147
it000 41437444 1200
it000 29506016 1271
it000 53693303 1292
it000 54217192 1307
it000 28913647 1331
it000 57383658 1421
it000 63632998 1478
it000 42422709 1482
it000 53519711 1488
it000 63040046 1494
it000 60528775 1531
it000 48935478 1565
it000 52628708 1597
it000 46987529 1604
it000 53710136 1709
it000 59836441 1777
it000 58494168 1820


KeyError: 5.626

In [70]:
it_id = "it000"

src_dat = proc_data[it_id]["source"]
ana_msk = src_dat["analyse_flag"][()] == 1

gcid = 22460446
gcid_idx = np.where(src_dat["gc_id"][ana_msk] == gcid)[0][0]
gcid_grp = src_dat["group_id"][ana_msk][gcid_idx]
gcid_grp

9106256

In [74]:
snp_dat_600 = proc_data[it_id]["snapshots"]["snap600"]
gc_sur_lst = snp_dat_600["gc_id"][()]

np.where(gc_sur_lst == gcid)[0][0]

snp_dat_600["group_id"][190]

9106256

In [71]:
ex_grp_lst

array([ 1920378,  4414957,  8580896, 13687078, 16199669, 19898495])

In [ ]:
def get_kde(x, y, xlim, ylim, grid_size=200):
    # Stack coordinates
    xy = np.vstack([x, y])

    # Compute KDE
    kde = stats.gaussian_kde(xy)

    # Create grid
    xmin, xmax = xlim
    ymin, ymax = ylim
    X, Y = np.mgrid[
        xmin : xmax : grid_size * 1j, ymin : ymax : grid_size * 1j
    ]  # use complex step for number of points
    positions = np.vstack([X.ravel(), Y.ravel()])

    # Evaluate KDE
    Z = np.reshape(kde(positions).T, X.shape)

    # Apply bounds mask
    mask = (X >= 0) & (Y >= 0)  # only allow x>=0, y>=0, y<=x
    Z_masked = np.where(mask, Z, 0)  # set values outside bounds to 0

    return Z_masked

In [ ]:
def contour_level_from_mass(Z, X, Y, frac=0.75):
    """
    Compute contour level that encloses `frac` of the total probability mass.
    Z : 2D KDE array
    X, Y : 2D meshgrid arrays
    frac : fraction of mass to enclose (0-1)
    """
    dx = X[1, 0] - X[0, 0]  # grid spacing in x
    dy = Y[0, 1] - Y[0, 0]  # grid spacing in y
    cell_area = dx * dy

    Z_flat = Z.ravel()
    Z_sort = np.sort(Z_flat)[::-1]  # sort descending
    cumsum = np.cumsum(Z_sort * cell_area)
    cumsum /= cumsum[-1]  # normalize to 1

    idx = np.searchsorted(cumsum, frac)
    return Z_sort[idx]